# Draft of the rule-based ensemble that combines submission files (precursor of `notebooks/05_ensemble_submission.ipynb`)

In [ ]:
DATA_DIR = "../data"

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
# read submission\submission_indobert_edu.csv and submission\Baseline.csv

submission = pd.read_csv(f'{DATA_DIR}/submission_indobert_edu.csv')

baseline = pd.read_csv(f'{DATA_DIR}/Baseline_2.csv')

In [ ]:
baseline

,IDText,label
0,TXT0001,Politik
1,TXT0002,Politik
2,TXT0003,Ideologi
3,TXT0004,Politik
4,TXT0005,Ekonomi
...,...,...
995,TXT0996,Ekonomi
996,TXT0997,Politik
997,TXT0998,Politik
998,TXT0999,Ekonomi


In [ ]:
submission

,IDText,text,Kelas
0,TXT0001,Lu mau orang 2 pro demokrasi di negara ini bis...,Politik
1,TXT0002,Prabowo ditanya soal hutang luar negeri dia me...,Pertahanan dan Keamanan
2,TXT0003,kiki daliyo Ganjar Pranowo itulah beliau sosok...,Ideologi
3,TXT0004,Prabowo Gibran yang bisa melakukan itu semua d...,Politik
4,TXT0005,Lah justru yang gak nyambung junjungan elu aom...,Sosial Budaya
...,...,...,...
995,TXT0996,Bikin bangga deh Ganjar Mahfud mau alokasikan ...,Ekonomi
996,TXT0997,Pak Jokowi sebelum pemilihan presiden 2024 ber...,Ideologi
997,TXT0998,Sbaiknya si gemot nga usah ikutan debat deh ng...,Sumber Daya Alam
998,TXT0999,kebiasaan merembuk atau bermusyawarah jadi gay...,Ekonomi


In [ ]:
baseline[baseline['IDText'] == 'TXT0704']

,IDText,label
703,TXT0704,Politik


In [ ]:
baseline[baseline['label'] == 'Geografi']

,IDText,label
23,TXT0024,Geografi
704,TXT0705,Geografi
728,TXT0729,Geografi
735,TXT0736,Geografi


In [ ]:
# get index from baseline where the prediction is Geografi

geografi_index = baseline[baseline['label'] == 'Geografi'].index

In [ ]:
submission.iloc[geografi_index]

,IDText,text,Kelas
23,TXT0024,Apalagi kita sebgai negara kepulauan Bagian pa...,Sumber Daya Alam
704,TXT0705,sia sia dong bgn ibu kota negara kl pusat ekon...,Geografi
728,TXT0729,Reklamasi Jokuwi Ahok itu 17 Pulau dihentikan ...,Sumber Daya Alam
735,TXT0736,Kalau memang masuk ke Banten ngapain Anies sam...,Sumber Daya Alam


In [ ]:
# for all geografi_index, replace the prediction with submission prediction

for i in geografi_index:
    submission['Kelas'][i] = baseline['label'][i]

In [ ]:
submission.value_counts('Kelas')

Kelas
Politik                    369
Ekonomi                    172
Ideologi                   167
Pertahanan dan Keamanan    138
Sosial Budaya               86
Sumber Daya Alam            41
Demografi                   20
Geografi                     7
dtype: int64

In [ ]:
submission[submission['IDText'] == 'TXT0705']

,IDText,text,Kelas
704,TXT0705,sia sia dong bgn ibu kota negara kl pusat ekon...,Geografi


In [ ]:
# print the row if "sapi" exist in the text of submission

sapi_idx = submission[submission['text'].str.contains('impor sapi')].index

In [ ]:
submission.loc[sapi_idx, 'Kelas'] = 'Sumber Daya Alam'

In [ ]:
submission[submission['text'].str.contains('impor sapi')]['Kelas']

720    Sumber Daya Alam
Name: Kelas, dtype: object

In [ ]:
mirip_all = pd.read_csv(f'{DATA_DIR}/mirip_all.csv')

In [ ]:
mirip_all

,IDText,text,text_embed,cosine_sim,train_text,train_label
0,TXT0001,lu mau org 2 pro demokrasi di negara ini bisa ...,[ 0.02898481 -0.0362686 -0.02264847 ... -0.00...,0.888701,partai politik mau ubah omnibuslaw ha ha kalia...,Politik
1,TXT0002,prabowo ditanya soal hutang luar negeri dia me...,[ 0.02540058 0.00145422 -0.02080927 ... -0.01...,0.885221,anies kehilangan otoritas bicara etika pada de...,Politik
2,TXT0003,kiki daliyo ganjar pranowo itulah beliau sosok...,[ 0.0037599 -0.01434812 -0.02095645 ... -0.01...,0.927224,kiki daliyo ganjar pranowo pak ganjar jadi ins...,Ideologi
3,TXT0004,prabowo gibran yang bisa melakukan itu semua d...,[ 0.03111109 -0.01025872 -0.01077099 ... -0.03...,0.929746,prabowo semangat untuk keadilan dan kesejahter...,Politik
4,TXT0005,lah justru yang gak nyambung junjungan elu aom...,[ 0.006633 -0.01374538 -0.00061216 ... -0.00...,0.895288,lihat bagian ganjar nanya anies soal nilai pas...,Sosial Budaya
...,...,...,...,...,...,...
995,TXT0996,bikin bangga deh ganjar mahfud mau alokasikan ...,[ 0.02377846 0.00020238 -0.00740114 ... -0.00...,0.915076,ganjar mahfud hebat bukti nyata kalau pak ganj...,Politik
996,TXT0997,pak jokowi sebelum pilpres 2024 berbesar hati ...,[ 0.0247977 0.00234677 -0.0230146 ... -0.01...,0.887695,rakyat pun paham sewaktu pencalonan mega prabo...,Politik
997,TXT0998,sbaiknya si gemot nga usah ikutan debat deh ng...,[ 0.03315755 -0.0061429 -0.04025121 ... -0.00...,0.915585,pak aniis dana apbd dki 80 t bagaimana dengan ...,Politik
998,TXT0999,kebiasaan merembuk atau bermusyawarah jadi gay...,[ 0.02118347 -0.01252864 -0.01274916 ... -0.01...,0.916667,pak ganjar akan hapus kredit macet petani dan ...,Politik


In [ ]:
# loop trough test_df_embed and submission
for i, row in mirip_all.iterrows():
    # print submission at index i
    # if mirip_all.loc[i, 'train_label'] == 'Politik' and mirip_all.loc[i, 'cosine_sim'] >= 0.98:
    #     submission.loc[i,'Kelas'] = 'Politik'
    # print(submission.loc[i]['label'])
    if mirip_all.loc[i, 'train_label'] == 'Ekonomi' and mirip_all.loc[i, 'cosine_sim'] >= 0.95:
        submission.loc[i,'Kelas'] = 'Ekonomi'
    elif mirip_all.loc[i, 'train_label'] == 'Geografi' and mirip_all.loc[i, 'cosine_sim'] >= 0.909849882125854:
        submission.loc[i,'Kelas'] = 'Geografi'
    elif mirip_all.loc[i, 'train_label'] == 'Ideologi' and mirip_all.loc[i, 'cosine_sim'] >= 0.938026428222656:
        submission.loc[i,'Kelas'] = 'Ideologi'
    elif mirip_all.loc[i, 'train_label'] == 'Pertahanan dan Keamanan' and mirip_all.loc[i, 'cosine_sim'] >= 0.952700495719909:
        submission.loc[i,'Kelas'] = 'Pertahanan dan Keamanan'
    elif mirip_all.loc[i, 'train_label'] == 'Sosial Budaya' and mirip_all.loc[i, 'cosine_sim'] >= 0.942158102989196:
        submission.loc[i,'Kelas'] = 'Sosial Budaya'

In [ ]:
submission.value_counts('Kelas')

Kelas
Politik                    366
Ekonomi                    173
Ideologi                   167
Pertahanan dan Keamanan    138
Sosial Budaya               89
Sumber Daya Alam            42
Demografi                   19
Geografi                     6
dtype: int64

In [ ]:
submission = submission[['IDText', 'Kelas']]

In [ ]:
submission.to_csv('/', index=False)

In [ ]:
submission_nopal = pd.read_csv(f'{DATA_DIR}/catboost_nopal.csv') # Catboost Biasa
submissionv2 = pd.read_csv(f'{DATA_DIR}/final78.csv') # indobert-edu + leakage
submissionijelid = pd.read_csv(f'{DATA_DIR}/submission_ijelid.csv') # Ijelid
submissionv1 = pd.read_csv(f'{DATA_DIR}/submission_gaming.csv') # Submission Before

In [ ]:
submission_nopal

,IDText,text,text_2,label_lemma,label_polos
0,TXT0001,lu mau orang 2 pro demokrasi di negara ini bis...,lu mau orang 2 pro demokrasi di negara ini bis...,Politik,Politik
1,TXT0002,prabowo ditanya soal hutang luar negeri dia me...,prabowo tanya soal hutang luar negeri dia jawa...,Pertahanan dan Keamanan,Politik
2,TXT0003,kiki daliyo ganjar pranowo itulah beliau sosok...,kiki daliyo ganjar pranowo itu beliau sosok ya...,Ideologi,Ideologi
3,TXT0004,prabowo gibran yang bisa melakukan itu semua d...,prabowo gibran yang bisa laku itu semua demi s...,Politik,Politik
4,TXT0005,lah justru yang gak nyambung junjungan elu aom...,lah justru yang gak nyambung junjung elu aomkm...,Ekonomi,Ekonomi
...,...,...,...,...,...
995,TXT0996,bikin bangga deh ganjar mahfud mau alokasikan ...,bikin bangga deh ganjar mahfud mau alokasi ten...,Ekonomi,Ekonomi
996,TXT0997,pak jokowi sebelum pemilihan presiden 2024 ber...,pak jokowi belum pilih presiden 2024 besar hat...,Politik,Politik
997,TXT0998,sbaiknya si gemot nga usah ikutan debat deh ng...,sbaiknya si got nga usah ikut debat deh nga je...,Politik,Politik
998,TXT0999,kebiasaan merembuk atau bermusyawarah jadi gay...,biasa rembuk atau musyawarah jadi gaya pimpin ...,Ekonomi,Ekonomi


In [ ]:
# check if Kelas is the same on submission_nopal, submissionijelid, and submissionv1
# check for class Ekonomi, Sosial Budaya, and Ideologi

for i, row in submission_nopal.iterrows():
    if submission_nopal.loc[i, 'label_polos'] == 'Ekonomi' and submissionijelid.loc[i, 'Kelas'] == 'Ekonomi' and submissionv1.loc[i, 'Kelas'] == 'Ekonomi':
        #change value in submissionv2
        if submissionv2.loc[i, 'Kelas'] != 'Demografi':
            submissionv2.loc[i, 'Kelas'] = 'Ekonomi'
    if submission_nopal.loc[i, 'label_polos'] == 'Sosial Budaya' and submissionijelid.loc[i, 'Kelas'] == 'Sosial Budaya' and submissionv1.loc[i, 'Kelas'] == 'Sosial Budaya':
        #change value in submissionv2
        if submissionv2.loc[i, 'Kelas'] != 'Demografi':
            submissionv2.loc[i, 'Kelas'] = 'Sosial Budaya'
    if submission_nopal.loc[i, 'label_polos'] == 'Ideologi' and submissionijelid.loc[i, 'Kelas'] == 'Ideologi' and submissionv1.loc[i, 'Kelas'] == 'Ideologi':
        #change value in submissionv2 but check if it is other than Demografi
        if submissionv2.loc[i, 'Kelas'] != 'Demografi':
            submissionv2.loc[i, 'Kelas'] = 'Ideologi'

In [ ]:
# loop trough test_df_embed and submissionv2
for i, row in mirip_all.iterrows():
    # print submissionv2 at index i
    if mirip_all.loc[i, 'train_label'] == 'Politik' and mirip_all.loc[i, 'cosine_sim'] >= 0.98:
        if submissionv2.loc[i,'Kelas'] != 'Demografi' and submissionv2.loc[i,'Kelas'] != 'Geografi' and submissionv2.loc[i,'Kelas'] != 'Sumber Daya Alam':
            submissionv2.loc[i,'Kelas'] = 'Politik'
    # print(submissionv2.loc[i]['label'])

In [ ]:
submissionv2.value_counts('Kelas')

Kelas
Politik                    400
Ekonomi                    162
Ideologi                   162
Pertahanan dan Keamanan    132
Sosial Budaya               79
Sumber Daya Alam            40
Demografi                   19
Geografi                     6
dtype: int64

In [ ]:
# search trough submission_nopal and check if contain 'bloomberg' in the text
bloomberg_idx = submission_nopal[submission_nopal['text'].str.contains('bloomberg')].index

In [ ]:
submission_nopal.loc[bloomberg_idx]

,IDText,text,text_2,label_lemma,label_polos
421,TXT0422,hasil survei media ekonomi bisnis berbasis di ...,hasil survei media ekonomi bisnis basis di ame...,Ekonomi,Ekonomi
597,TXT0598,media ekonomi bisnis berbasis di amerika serik...,media ekonomi bisnis basis di amerika serikat ...,Ekonomi,Ekonomi
648,TXT0649,survei bloomberg unggulkan anies di pemilihan ...,survei bloomberg unggul anies di pilih preside...,Politik,Politik
969,TXT0970,survei media ekonomi bisnis terkemuka asal as ...,survei media ekonomi bisnis muka asal as bloom...,Ekonomi,Ekonomi


In [ ]:
submissionv2.loc[bloomberg_idx, 'Kelas'] = 'Politik'

In [ ]:
submissionv2.value_counts('Kelas')

Kelas
Politik                    402
Ideologi                   162
Ekonomi                    160
Pertahanan dan Keamanan    132
Sosial Budaya               79
Sumber Daya Alam            40
Demografi                   19
Geografi                     6
dtype: int64

In [ ]:
submissionv2.to_csv(f'{DATA_DIR}/elngide.csv', index=False)

In [ ]:
for i, row in submissionv2.iterrows():
    if submissionv2.loc[i,'Kelas'] == 'Sumber Daya Alam' and submissionv1.loc[i, 'Kelas'] == 'Sosial Budaya' and submissionijelid.loc[i, 'Kelas'] == 'Sosial Budaya':
        print('x')

x
x
x
x
x
x
